# பாடம் 12 - முகவர் ஸ்க்ராட்ச்பாட் மூலம் உரையாடல் வரலாற்று குறைக்கல்

இந்த நோட்புக்Microsoft Agent Framework ஐ பயன்படுத்தி நீண்ட உரையாடல்களில் பொருள் வினியோகிப்பை எவ்வாறு நிர்வகிப்பது என்பதை விளக்குகிறது. உரையாடல்கள் வளரும்போது, டோக்கன் எண்ணிக்கை அதிகரிக்கும் — இறுதியில் மாதிரியின் பொருள் சாளரம் மீறிவிடும். இதை நாம் **பொருள் சுருக்க முறைமையால்** மற்றும் நிலையான நினைவுக்கான **முகவர் ஸ்க்ராட்ச்பாட்** மூலம் கையாள்கிறோம்.

## நீங்கள் கற்கப்போகும் விஷயங்கள்:
1. **ஏன் பொருள் நிர்வாகம் அவசியம்**: டோக்கன் வரம்புகள் மற்றும் பொருள் சாளரங்களைப் புரிந்துகொள்வது
2. **பொருள் உணர்ந்த முகவர்கள்**: தங்களுடைய உரையாடல் பொருளை நிர்வகிக்கும் முகவர்களை உருவாக்குதல்
3. **பொருள் சுருக்க முறைமை**: உரையாடல் வரலாறை சுருக்கக் கருவிகள் பயன்பாடு
4. **முகவர் ஸ்க்ராட்ச்பாட்**: பொருள் குறைப்பின்பாலும் நிலைத்த நினைவகம்

## முன்னாசிரியத் தேவைகள்:
- சுற்றுச்சூழல் மாறிகளை உள்ளமைக்க Azure OpenAI அமைப்பு
- முந்தைய பாடங்களில் அடிப்படைக் கூரைகள் பற்றிய புரிதல்


## செட்டப்


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## சூழல் மேலாண்மை ஏன் முக்கியம்

ஒவ்வொரு LLMக்கும் ஒரு நிரந்தரமான **சூழல் ஜன்னல்** உள்ளது — ஒரே கோரிக்கையில் அது செயலாக்கக்கூடிய அதிகபட்ச டோக்கன் எண்ணிக்கை. பல முறை உரையாடல் முன்னேறும்போது:

- **டோக்கன் எண்ணிக்கை ஒவ்வொரு பயனர் செய்தி மற்றும் உதவியாளர் பதிலுடன் வரிசையாக அதிகரிக்கிறது**.
- **சொல்லிக்கை டோக்கன்கள் செலவை அதிகமாக்குகின்றன** ஏனெனில் முழுமையான வரலாறு ஒவ்வொரு முறையும் மீண்டும் அனுப்பப்படுகிறது.
- இறுதி நிலைக்கு உரையாடல் **சூழல் ஜன்னலை மீறுகிறது** மற்றும் மாதிரி அல்லது குறைக்கிறது அல்லது பிழை ஏற்படுகிறது.

### சூழலை மேலாண்மை செய்யும் 策略கள்

| 策略 | அது எப்படி செயல்படுகிறது | பிணியாக்கம் |
|---|---|---|
| **குறைத்தல்** | பழைய செய்திகளை தவிர்த்தல் | ஆரம்ப சூழலை இழக்கிறது |
| **சுருக்கல்** | பழைய செய்திகள் சுருக்கமாக தொகுக்கப்படுகிறது | சில தகவல்கள் இழக்கப்பட்டாலும் முக்கிய விவரங்கள் பாதுகாக்கப்படுகிறது |
| **வரைவகம் / வெளிப்புற நினைவகம்** | உரையாடல் வெளியில் முக்கிய உண்மைகள் சேமிக்கப்படுகிறது | கருவி அழைப்புகள் தேவை, ஆனால் எந்த குறைக்கும் இருப்பினும் வாழும் |

இந்த நோட்புக் இல் நாம் **சுருக்கல்** மற்றும் **வரைவகம் கருவி** ஐ இணைத்து, உரையாடல் வரலாறு சுருக்கப்பட்டாலும் முகவர் தொடர்ச்சியை பராமரிக்க முடியும்.


## ஒரு சூழ்நிலை-அறிந்த முகவரை உருவாக்குதல்


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## நீண்ட உரையாடலை சிக்கலாய் நிகழ்த்தல்

சூழல் எவ்வாறு சேர்கிறது என்பதை காண பலமுறை உரையாடல் நடைமுறையில் செல்லலாம். முகவர் முக்கிய விபரங்களை (மனப்பான்மைகள், பட்ஜெட், பயணத் தேதி) முறைசார்ந்த உரையாடல்களில் நினைவில் வைக்க வேண்டும் மற்றும் தொடர்ச்சியை காட்ட வேண்டும்.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

முகவர் ஏற்கனவே நடந்த உரையாடலிலிருந்த தகவல்களை எவ்வாறு வைத்திருக்கிறாரோ கவனியுங்கள் — ஜப்பான், சூஷி, கோயில்கள், புகைப்படம், $3000 பட்ஜெட், தனியாகப் பயணம், மற்றும் ஏப்ரல் காலம் ஆகியவற்றை தெரிந்துகொண்டிருக்கிறார். ஒரு குறுகிய உரையாடலில் இது நன்றாக வேலை செய்கிறது, ஆனால் உரையாடல் நீண்டபோது முழு வரலாறையோடு மீண்டும் அனுப்புவது திடீரென்று செலவாகிறது.

உரையாடல் தொடர்ந்து விரிவடையும் போது தகவல் சேகரிப்பைக் காணலாம்:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## உரையாடல் சுருக்கல் மாதிரி

உரையாடல் வளர்வதற்கு, நாம் **சுருக்கல் கருவி** பயன்படுத்தி சேகரித்துள்ள பின்னணியை சுருக்கமான வடிவில் ஒருங்கிணைக்கலாம். முகவர் இந்த கருவியை முக்கிய விருப்பங்களை பதிவு செய்ய அழைக்கிறார், இதனால் பழைய செய்திகள் எதுவும் நீக்கப்பட்டாலும், அவசியமான தகவல்கள் பாதுகாக்கப்படும்.

இந்த மாதிரி அதிக நுணுக்கமான வரலாறு குறைத்தல் செயலின் அடிப்படையாகிறது:
1. முகவர் உரையாடலிலிருந்து முக்கிய உண்மைகள் கண்டறிகிறார்
2. அவற்றைப் பதிவிற்கு சுருக்கல் கருவியை அழைக்கிறார்
3. பழைய செய்திகள் பாதுகாப்பாக நீக்கப்படலாம், ஏனெனில் சுருக்கம் முக்கியமானவை மீட்டெடுக்கிறது

கீழே உங்கள் முகவர் கற்றுக்கொண்டவற்றின் சுருக்கமான உறுப்படியைப் பதிவு செய்ய `summarize_preferences` கருவியை வரையறுக்கின்றோம்.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

इस पाठ में आपने Microsoft Agent Framework का उपयोग करके लंबी अवधि चलने वाली एजेंट वार्तालापों में संदर्भ प्रबंधन करना सीखा:

### मुख्य अवधारणाएँ
- **संदर्भ विंडो सीमित होती हैं** — वार्तालाप इतिहास में हर टोकन की लागत होती है और यह सीमा की ओर गिना जाता है।
- **संक्षेपण उपकरण** एजेंट को संचयित संदर्भ को संक्षिप्त सारांशों में संपीड़ित करने की अनुमति देते हैं, टोकन उपयोग को कम करते हुए आवश्यक जानकारी संरक्षित रखते हैं।
- **एजेंट स्क्रैचपैड** निरंतर बाह्य स्मृति प्रदान करते हैं जो किसी भी वार्तालाप कटौती से बच जाती है।

### आपने क्या बनाया
- एक **संदर्भ-सजग एजेंट** जो बहु-चरणीय वार्तालापों में निरंतरता बनाए रखता है
- एक **संक्षेपण उपकरण** (`summarize_preferences`) जो उपयोगकर्ता के प्रमुख विवरणों को संक्षिप्त प्रारूप में रिकॉर्ड करता है
- एक **बहु-चरणीय वार्तालाप** जो संदर्भ संरक्षण और परिवर्तन प्रबंधन को प्रदर्शित करता है

### वास्तविक-विश्व अनुप्रयोग
- **ग्राहक सेवा बॉट्स**: लंबे समर्थन सत्रों के दौरान प्राथमिकताओं को याद रखें
- **व्यक्तिगत सहायक**: संदर्भ को दोबारा समझाए बिना चालू परियोजनाओं को ट्रैक करें
- **शैक्षिक ट्यूटर**: कई इंटरैक्शन के दौरान छात्र की प्रगति बनाए रखें

### अगले चरण
- फ़ाइल-आधारित स्थिरता के साथ पूर्ण स्क्रैचपैड उपकरण लागू करें
- संक्षेपण के बाद स्वचालित इतिहास ट्रंकेशन जोड़ें
- अर्थपूर्ण स्मृति खोज के लिए वेक्टर डेटाबेस के साथ संयोजन करें
- पूर्ण संदर्भ के साथ दिन बाद वार्तालाप फिर से शुरू करने वाले एजेंट बनाएं


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
